# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HarveyWebbs/ML-Basics/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*This task is framed as a Regression (Scoring) problem. Instead of classifying content into arbitrary buckets like "good" or "bad", the goal is to predict a continuous numeric output (traffic/visibility). By training a regression model (or running multivariate regression analysis), we can look at the weights (coefficients) or feature importances assigned to each signal. This tells us not only which signals matter, but how much they matter in relation to one another.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*WTarget: total_clicks (or total_impressions) aggregated over a specific time window.
Proxy context: Because web traffic follows a heavy power-law distribution (where a few pages get millions of clicks and most get zero), the raw click count is highly skewed. To make this a viable ML regression target, I will likely use a log-transformed proxy: log(total_clicks + 1). This normalizes the data so the model doesn't over-index entirely on the top 1% of viral articles.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric

*Offline ML Metric: R-Squared. R-squared will tell us what percentage of the variance in traffic can actually be explained by the safe content signals we have.
Business Metric: The extraction of actionable coefficients. Success is handing the content team a prioritized list of signals with estimated effect sizes.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

*One row = One unique pseudonymized content item.*

In [3]:
import duckdb

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")

# Make sure your real token is here!
con.execute("CREATE OR REPLACE SECRET (TYPE huggingface, TOKEN '')")

rel = "hf://datasets/FlyRank/internship-warehouse"

# FIX: Using 'gsc_impressions' and 'gsc_clicks' based on the real schema
query = f"""
SELECT
    c.content_hash_id,
    c.content_type,
    c.content_created_date,
    c.content_updated_date,
    SUM(p.gsc_impressions) as total_gsc_impressions,
    SUM(p.gsc_clicks) as total_gsc_clicks
FROM read_parquet('{rel}/dim_content.parquet') c
JOIN (
    -- Limiting to 100k rows of the 78M row daily performance table
    SELECT content_hash_id, gsc_impressions, gsc_clicks
    FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
    LIMIT 100000
) p ON c.content_hash_id = p.content_hash_id
GROUP BY 1, 2, 3, 4
LIMIT 5
"""

unit_of_analysis_df = con.sql(query).df()
print("Unit of Analysis: ONE ROW = ONE PIECE OF CONTENT")
display(unit_of_analysis_df)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Unit of Analysis: ONE ROW = ONE PIECE OF CONTENT


,content_hash_id,content_type,content_created_date,content_updated_date,total_gsc_impressions,total_gsc_clicks
0,content_746054f8c9302fa4,keyword article,2025-02-12,2026-05-18,67.0,0.0
1,content_7468d96a992d9939,keyword article,2025-03-03,2026-05-18,29.0,0.0
2,content_7476254360f720e6,keyword article,2025-03-03,2026-05-18,11.0,0.0
3,content_748265245917d0c5,keyword article,2025-02-12,2026-05-18,15.0,0.0
4,content_749a21e42bf7e009,keyword article,2025-02-12,2026-06-24,1684.0,2.0


## 5. Why ML beats a fixed rule here

*The pattern of SEO traffic is far too messy for hardcoded rules for three main reasons:
Complex Feature Interactions: Signals do not exist in a vacuum. A "News" article might lose 99% of its traffic if it hasn't been updated in a year, whereas a "Definition" or "Glossary" page might retain its traffic for 5 years without ever being updated. An if-statement cannot easily capture how content_type dynamically changes the importance of content_updated_date.
Non-Linearity & Diminishing Returns: Hard cut-offs fail in continuous distributions. If our rule says "updated within 6 months is good," it treats a 5-month-old article and a 1-day-old article as exactly the same. Machine learning regression can capture the actual decay curves—showing exactly how traffic tapers off day-by-day.
Scale of Variables: As we add more dimensions (competition, keyword difficulty, internal linking), human-written if-statements become impossible to balance. An ML model (like a Gradient Boosting Regressor) mathematically weighs these competing signals against one another to find the most probable outcome*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.